In [1]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import json
from ast import literal_eval


In [2]:
credits = pd.read_csv('movie_dataset/credits.csv')
keywords = pd.read_csv('movie_dataset/keywords.csv')
metadata = pd.read_csv('movie_dataset/movies_metadata.csv')
links_small = pd.read_csv('movie_dataset/links_small.csv')
ratings_small = pd.read_csv('movie_dataset/ratings_small.csv')
# links = pd.read_csv('movie_dataset/links.csv')
# ratings = pd.read_csv('movie_dataset/ratings.csv')

In [3]:
credits['cast'] = credits['cast'].apply(literal_eval)
credits['crew'] = credits['crew'].apply(literal_eval)

keywords['keywords'] = keywords['keywords'].apply(literal_eval)

links_small['tmdbId'] = links_small[links_small['tmdbId'].notnull()]['tmdbId'].astype('int')

metadata["id"] = pd.to_numeric(metadata["id"], errors='coerce') #Invalid parsing set to NaN
metadata = metadata.dropna(subset=['id'])
metadata["id"] = metadata["id"].astype(int)
metadata['genres'] = metadata['genres'].apply(literal_eval)

ratings_small['movieId'] = ratings_small[ratings_small['movieId'].notnull()]['movieId'].astype('int')
ratings_small['userId'] = ratings_small[ratings_small['userId'].notnull()]['userId'].astype('int')
ratings_small.drop(columns=['timestamp'], inplace=True)

In [4]:
md = metadata.merge(credits, on='id')
md = md.merge(keywords, on='id')
md = md[md["id"].isin(links_small['tmdbId'])]

md.drop([
        "adult",
        "belongs_to_collection", "budget", "homepage", "original_language", "original_title", "overview",
        "popularity", "poster_path", "production_companies", "production_countries", "release_date", 
        "revenue", "runtime", 'spoken_languages', 'status', 'tagline', 'video',
        'vote_average', 'vote_count'
        ], axis=1, inplace=True)
md.columns

Index(['genres', 'id', 'imdb_id', 'title', 'cast', 'crew', 'keywords'], dtype='object')

In [5]:
md['cast'] = md['cast'].apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else []) # non-list values to empty list

md['cast'] = md['cast'].apply(lambda x: [str.lower(i.replace(" ", "")) for i in x])

md['keywords'] = md['keywords'].apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])

md['director'] = md['crew'].apply(lambda x: [i['name'] for i in x if i['job'] == 'Director'] if isinstance(x, list) \
                                  and any(i['job'] == 'Director' for i in x) else None)

md['director'] = md['director'].apply(lambda x: [str.lower(i.replace(" ", "")) for i in x] if isinstance(x, list) else [])

md['genres'] = md['genres'].apply(lambda x: [i['name'].lower() for i in x] if isinstance(x, list) else [])



In [6]:
avg_ratings = ratings_small.groupby('movieId', as_index=False)['rating'].mean().round(2)
md = md.merge(avg_ratings, left_on='id', right_on='movieId', how='left')

In [7]:
md.columns

Index(['genres', 'id', 'imdb_id', 'title', 'cast', 'crew', 'keywords',
       'director', 'movieId', 'rating'],
      dtype='object')

In [8]:
ratings_small.columns

Index(['userId', 'movieId', 'rating'], dtype='object')

# Content Based


In [9]:
md['soup'] = md['keywords'] + md['cast'] + md['director'] + md['genres']
md['soup'] = md['soup'].apply(lambda x: ' '.join(x))

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem.snowball import SnowballStemmer

In [11]:
stemmer = SnowballStemmer('english')
countvec = CountVectorizer(analyzer='word',ngram_range=(1, 2), stop_words='english')

In [12]:
md['keywords'] = md['keywords'].apply(lambda x: [stemmer.stem(i) for i in x])
# md['keywords'] = md['keywords'].apply(lambda x: [str.lower(i.replace(" ", "")) for i in x])

In [13]:
count_matrix = countvec.fit_transform(md['soup'])
cosine_sim = cosine_similarity(count_matrix, count_matrix)

In [14]:
md = md.reset_index()
titles = md['title']
indices = pd.Series(md.index, index=md['title'])

In [15]:
def get_recommendations(title):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:31]
    movie_indices = [i[0] for i in sim_scores]
    return titles.iloc[movie_indices]

In [16]:
get_recommendations('Timecop').head(10)

1211                     Double Team
1674                            Tron
7086    20,000 Leagues Under the Sea
216                 Gumby: The Movie
7639                          TEKKEN
1168                     The Machine
9117               World of Tomorrow
2595                          Poison
2101              My Science Project
5548    The Three Lives of Thomasina
Name: title, dtype: object

In [17]:
get_recommendations('Inception').head(10)

5548             The Three Lives of Thomasina
3114                      Highlander: Endgame
343                                   Timecop
3934                The Count of Monte Cristo
5574                      Yu-Gi-Oh! The Movie
5602    Sky Captain and the World of Tomorrow
6292            Left Behind III: World at War
7319              G.I. Joe: The Rise of Cobra
5264                                   Zardoz
7086             20,000 Leagues Under the Sea
Name: title, dtype: object

# Collaborative

In [18]:
from sklearn.decomposition import TruncatedSVD

In [19]:
ratings_movie_ids = set(ratings_small['movieId'])
metadata_movie_ids = set(metadata['id'])

print(f"Ratings has {len(ratings_movie_ids)} unique movies")
print(f"Metadata has {len(metadata_movie_ids)} unique movies")
print(f"Overlap: {len(ratings_movie_ids & metadata_movie_ids)} movies")

Ratings has 9066 unique movies
Metadata has 45433 unique movies
Overlap: 2830 movies


In [20]:
# Keep only movies that exist in both
common_movies = ratings_movie_ids & metadata_movie_ids

ratings_filtered = ratings_small[ratings_small['movieId'].isin(common_movies)]
metadata_filtered = metadata[metadata['id'].isin(common_movies)]

# Rebuild rating matrix
rating_matrix = ratings_filtered.pivot_table(
    index='userId', 
    columns='movieId', 
    values='rating',
    fill_value=0
)

# Retrain SVD
svd = TruncatedSVD(n_components=50, random_state=42)
latent_factors = svd.fit_transform(rating_matrix)
user_similarities = cosine_similarity(latent_factors)



In [21]:
def recommend_cf(user_id, n=5, ratings_df=ratings_small, rating_matrix=rating_matrix, user_similarities=user_similarities):
    if user_id not in rating_matrix.index:
        return []
    ## get user index and  similarities with other users
    user_idx = rating_matrix.index.get_loc(user_id)
    similar_users = user_similarities[user_idx]
    
    ## get movies already rated by user
    user_rated = set(ratings_df[ratings_df['userId'] == user_id]['movieId'])
    
    scores = {}
    ## iterate over all movies
    for movie_id in rating_matrix.columns:
        ## if user has not rated that movie
        if movie_id not in user_rated:
            movie_ratings = ratings_df[ratings_df['movieId'] == movie_id]
            ## for all ratings for that movie
            for _, row in movie_ratings.iterrows():
                ##  check for other users who rated that movie
                if row['userId'] in rating_matrix.index:
                    other_user_idx = rating_matrix.index.get_loc(row['userId'])
                    ## get score for every user and movie combination based on similarity and rating
                    score = similar_users[other_user_idx] * row['rating']
                    scores[movie_id] = scores.get(movie_id, 0) + score
    
    top_movies = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n] ## get top n movies based on score
    return top_movies

In [22]:
# Now recommend
recs = recommend_cf(user_id=2, n=5, ratings_df=ratings_filtered)
for movie_id, score in recs:
    title = metadata_filtered[metadata_filtered['id'] == movie_id]['title'].values[0]
    print(f"{title}: {score:.2f}")

The Million Dollar Hotel: 482.79
Rain Man: 351.23
To Kill a Mockingbird: 320.83
The 39 Steps: 305.54
Men in Black II: 264.07
